# Training a Small Language Model (TinyGPT) on a Technical QA Dataset

This notebook performs hyperparameter search for a lightweight Transformer model (`TinyGPT`) and evaluates it on a QA dataset generated from a product manual. The goal is to find the best trade-off between model size (footprint on embedded devices) and performance (loss, perplexity, BLEU, exact match).

## Notebook Structure
1. **Imports and setup** – load required libraries and custom modules.
2. **Load and split the QA dataset** – use `QADataset` to load JSONL data.
3. **Hyperparameter search loop** – train many model configurations and collect metrics.
4. **Evaluation and analysis** – generate summary CSV, Pareto frontier, parallel coordinates plots, and select the best model.

All code is kept as in the original; only comments and printed messages have been translated to English, and explanatory markdown cells have been added.

In [ ]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
import os
import json
import itertools
import csv
from typing import Dict, List, Any
from src.slm import TinyGPT
from src.utils.dataset import QADataset
from sklearn.model_selection import train_test_split

## 2. Load and Prepare the QA Dataset
The dataset was generated in the previous notebook. We load it from a JSONL file and split into train/test sets (80/20).

In [ ]:
synthetic_dataset = QADataset().load_jsonl_file('dataset_CAT-4083-UK.json')
train_dataset, test_dataset = train_test_split(
    synthetic_dataset,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
train_dataset

## 3. Hyperparameter Search

We define a grid of hyperparameters (embedding dimension, number of attention heads, transformer layers, dropout rate, epochs, batch size, learning rate) and train a separate model for each combination. For each configuration we:
- Instantiate the TinyGPT model.
- Train it on the training set.
- Save model weights and generate Arduino-compatible C++ code.
- Evaluate on train and test sets.
- Collect metrics (loss, perplexity, token accuracy, BLEU, exact match) and model size information.

All results are saved to JSON and CSV files for later analysis.

In [ ]:
import itertools
import json
import os
import csv
from typing import Dict, List, Any

# ============================================================
# Helper functions for counting parameters and size
# ============================================================
def flatten_json_to_list(input_json_path: str) -> List[float]:
    """Extracts all numeric values from a JSON file recursively."""
    with open(input_json_path, 'r') as f:
        json_data = json.load(f)
    values = []
    
    def extract_values(data):
        if isinstance(data, dict):
            for key, value in data.items():
                extract_values(value)
        elif isinstance(data, list):
            for item in data:
                if isinstance(item, list):
                    for num in item:
                        if isinstance(num, (int, float)):
                            values.append(num)
                elif isinstance(item, (int, float)):
                    values.append(item)
        elif isinstance(data, (int, float)):
            values.append(data)
        elif data is None:
            pass  # Ignore None values
        else:
            raise ValueError(f"Unsupported type: {type(data)}")
    
    extract_values(json_data)
    return values

def count_total_parameters(path_input: str) -> int:
    """
    Returns the total number of numeric parameters by summing the files:
    embedding.json, final_norm.json, out_head.json and transformer_blocks.json.
    """
    file_names = [
        "embedding.json",
        "final_norm.json",
        "out_head.json",
        "transformer_blocks.json"
    ]
    
    total_params = 0
    
    for file_name in file_names:
        file_path = os.path.join(path_input, file_name)
        try:
            flattened = flatten_json_to_list(input_json_path=file_path)
            total_params += len(flattened)
        except FileNotFoundError:
            print(f"Warning: file not found: {file_name}")
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
    
    return total_params

def get_total_cpp_header_size(folder_path: str) -> int:
    """
    Returns the total size (in bytes) of .cpp and .h files in the specified folder.
    """
    total_size = 0
    if not os.path.exists(folder_path):
        print(f"Warning: directory not found: {folder_path}")
        return 0
    for file_name in os.listdir(folder_path):
        if file_name.endswith(('.cpp', '.h')):
            file_path = os.path.join(folder_path, file_name)
            total_size += os.path.getsize(file_path)
    return total_size

# ============================================================
# Assuming these classes and datasets are already available:
# from your_module import TinyGPT, AdamW, CrossEntropyLoss
# train_dataset, test_dataset = ...
# ============================================================

# ============================================================
# Definition of the hyperparameter grid to be tested
# ============================================================
param_grid = {
    'embedding_dim': [12, 24],
    'heads': [2, 4],
    'layers': [1, 2],
    'drop_rate': [0.01, 0.05],
    'num_epochs': [300, 500],
    'batch_size': [4, 8],
    'lr': [0.01, 0.005],
    # optimizer_cls and loss_fn_cls are fixed, but could be varied if desired
}

# Generate all combinations
keys = param_grid.keys()
values = param_grid.values()
combinations = list(itertools.product(*values))

# List to store metrics from all runs
all_results: List[Dict[str, Any]] = []

# ============================================================
# Main loop over combinations
# ============================================================
for idx, combo in enumerate(combinations):
    # Create parameter dictionary for this run
    params = dict(zip(keys, combo))
    
    # Generate a unique identifier for this configuration
    config_id = (
        f"emb{params['embedding_dim']}_"
        f"h{params['heads']}_"
        f"l{params['layers']}_"
        f"d{str(params['drop_rate']).replace('.', '')}_"
        f"ep{params['num_epochs']}_"
        f"bs{params['batch_size']}_"
        f"lr{str(params['lr']).replace('.', '')}"
    )
    
    print(f"\n{'='*60}")
    print(f"Running configuration {idx+1}/{len(combinations)}: {config_id}")
    print(f"Parameters: {params}")
    print(f"{'='*60}")
    
    # Create specific directories for this configuration
    output_model_dir = f"output_dir_{config_id}"
    output_arduino_dir = f"output_arduino_{config_id}"
    os.makedirs(output_model_dir, exist_ok=True)
    os.makedirs(output_arduino_dir, exist_ok=True)
    
    # Instantiate the model with the iteration's hyperparameters
    model = TinyGPT(
        embedding_dim=params['embedding_dim'],
        heads=params['heads'],
        layers=params['layers'],
        drop_rate=params['drop_rate']
    )
    
    # Train the model
    model.train(
        synthetic_dataset=train_dataset,
        num_epochs=params['num_epochs'],
        batch_size=params['batch_size'],
        lr=params['lr'],
        optimizer_cls=AdamW,          # Can be parameterized if needed
        loss_fn_cls=CrossEntropyLoss  # Can be parameterized if needed
    )
    
    # Save model and generate Arduino code
    model.save(output_folder=output_model_dir)
    model.code_generator(
        model_folder=output_model_dir,
        output_arduino_folder=output_arduino_dir,
        model_name=f"model_{config_id}"
    )
    
    # Evaluate on the training set
    train_metrics = model.evaluation(
        train_dataset,
        batch_size=32,  # evaluation batch size can be fixed or variable
        num_examples=len(train_dataset)
    )
    
    # Evaluate on the test set
    test_metrics = model.evaluation(
        test_dataset,
        batch_size=32,
        num_examples=len(test_dataset)
    )
    
    # Save metrics to configuration-specific JSON files
    train_json_path = f"evaluation_train_{config_id}.json"
    test_json_path = f"evaluation_test_{config_id}.json"
    
    with open(train_json_path, "w", encoding="utf-8") as f:
        json.dump(train_metrics, f, indent=2, ensure_ascii=False)
    with open(test_json_path, "w", encoding="utf-8") as f:
        json.dump(test_metrics, f, indent=2, ensure_ascii=False)
    
    # Calculate number of model parameters
    num_params = count_total_parameters(output_model_dir)
    
    # Calculate size of generated C++ code
    cpp_size = get_total_cpp_header_size(output_arduino_dir)
    
    # Store aggregated result for later comparison
    result_entry = {
        "config_id": config_id,
        "params": params,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "train_json": train_json_path,
        "test_json": test_json_path,
        "model_dir": output_model_dir,
        "arduino_dir": output_arduino_dir,
        "number_parameter": num_params,
        "cpp_model_size": cpp_size
    }
    all_results.append(result_entry)
    
    # Display main metrics in console
    print(f"Train - Avg loss: {train_metrics.get('avg_loss', 'N/A'):.4f}, Token accuracy: {train_metrics.get('token_accuracy', 'N/A'):.4f}")
    print(f"Test  - Avg loss: {test_metrics.get('avg_loss', 'N/A'):.4f}, Token accuracy: {test_metrics.get('token_accuracy', 'N/A'):.4f}")
    print(f"Model parameters: {num_params}")
    print(f"C++ code size: {cpp_size} bytes")

# ============================================================
# Save overall summary of all runs
# ============================================================
summary_path = "hyperparameter_search_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

print(f"\nHyperparameter search completed! Results saved to '{summary_path}'.")

# ============================================================
# Create CSV with detailed train and test metrics
# ============================================================
csv_path = "hyperparameter_search_summary.csv"
with open(csv_path, "w", newline='', encoding="utf-8") as csvfile:
    fieldnames = [
        "config_id",
        "embedding_dim", "heads", "layers", "drop_rate",
        "num_epochs", "batch_size", "lr",
        # Train metrics
        "train_avg_loss", "train_perplexity", "train_token_accuracy",
        "train_num_samples", "train_avg_bleu", "train_exact_match_rate",
        # Test metrics
        "test_avg_loss", "test_perplexity", "test_token_accuracy",
        "test_num_samples", "test_avg_bleu", "test_exact_match_rate",
        # Model size metrics
        "number_parameter", "cpp_model_size"
    ]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for entry in all_results:
        train = entry["train_metrics"]
        test = entry["test_metrics"]

        row = {
            "config_id": entry["config_id"],
            "embedding_dim": entry["params"]["embedding_dim"],
            "heads": entry["params"]["heads"],
            "layers": entry["params"]["layers"],
            "drop_rate": entry["params"]["drop_rate"],
            "num_epochs": entry["params"]["num_epochs"],
            "batch_size": entry["params"]["batch_size"],
            "lr": entry["params"]["lr"],
            # Train
            "train_avg_loss": train.get("avg_loss", ""),
            "train_perplexity": train.get("perplexity", ""),
            "train_token_accuracy": train.get("token_accuracy", ""),
            "train_num_samples": train.get("num_samples", ""),
            "train_avg_bleu": train.get("avg_bleu", ""),
            "train_exact_match_rate": train.get("exact_match_rate", ""),
            # Test
            "test_avg_loss": test.get("avg_loss", ""),
            "test_perplexity": test.get("perplexity", ""),
            "test_token_accuracy": test.get("token_accuracy", ""),
            "test_num_samples": test.get("num_samples", ""),
            "test_avg_bleu": test.get("avg_bleu", ""),
            "test_exact_match_rate": test.get("exact_match_rate", ""),
            # Model
            "number_parameter": entry["number_parameter"],
            "cpp_model_size": entry["cpp_model_size"]
        }
        writer.writerow(row)

print(f"Tabular summary saved to '{csv_path}'.")